In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.ensemble import RandomForestRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.svm import SVR
from sklearn.naive_bayes import GaussianNB
from sklearn.neural_network import MLPRegressor
import matplotlib.pyplot as plt
import seaborn as sns


In [ ]:
# ===============================
# Hybrid Model for Student Performance Prediction (FAST VERSION)
# ===============================

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.ensemble import RandomForestRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.svm import SVR
from sklearn.naive_bayes import GaussianNB
from sklearn.neural_network import MLPRegressor
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv("student-por.csv")
df.fillna(df.median(numeric_only=True), inplace=True)

encoder = LabelEncoder()
for col in df.select_dtypes(include="object").columns:
    df[col] = encoder.fit_transform(df[col])

X = df.drop("G3", axis=1)
y = df["G3"]

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42
)

print("Training Random Forest...")
rf = RandomForestRegressor(n_estimators=50, random_state=42)
rf.fit(X_train, y_train)
rf_pred = rf.predict(X_test)

importance = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False)

plt.figure(figsize=(10,6))
sns.barplot(x=importance[:10], y=importance.index[:10])
plt.tight_layout()
plt.show(block=False)

top_features = importance.index[:10]
X_top = scaler.fit_transform(df[top_features])

X_train, X_test, y_train, y_test = train_test_split(
    X_top, y, test_size=0.2, random_state=42
)

print("Training Decision Tree...")
dt = DecisionTreeRegressor(random_state=42)
dt.fit(X_train, y_train)
dt_pred = dt.predict(X_test)

print("Training SVM...")
svm = SVR(kernel='linear')
svm.fit(X_train, y_train)
svm_pred = svm.predict(X_test)

print("Training Naive Bayes...")
nb = GaussianNB()
nb.fit(X_train, y_train)
nb_pred = nb.predict(X_test)

print("Training MLP...")
mlp = MLPRegressor(hidden_layer_sizes=(30,), max_iter=300, random_state=42)
mlp.fit(X_train, y_train)
mlp_pred = mlp.predict(X_test)

print("Training Hybrid Model...")
rf_top = RandomForestRegressor(n_estimators=50, random_state=42)
rf_top.fit(X_train, y_train)

X_train_h = np.column_stack((X_train, rf_top.predict(X_train)))
X_test_h = np.column_stack((X_test, rf_top.predict(X_test)))

hybrid = MLPRegressor(hidden_layer_sizes=(30,), max_iter=300, random_state=42)
hybrid.fit(X_train_h, y_train)
hybrid_pred = hybrid.predict(X_test_h)

print("\nRESULTS:")
print("RF MAE:", mean_absolute_error(y_test, rf_pred))
print("DT MAE:", mean_absolute_error(y_test, dt_pred))
print("SVM MAE:", mean_absolute_error(y_test, svm_pred))
print("NB MAE:", mean_absolute_error(y_test, nb_pred))
print("MLP MAE:", mean_absolute_error(y_test, mlp_pred))
print("Hybrid MAE:", mean_absolute_error(y_test, hybrid_pred))
